# A5 polygon polyfill (`polygon2a5_new`)

Step-by-step animation matching vgrid [`polygon2a5_new`](https://github.com/opengeoshub/vgrid/blob/main/vgrid/conversion/vector2dggs/vector2a5.py) — `polygon_to_cells` + `uncompact` per part (not bbox BFS).

Input: [`multipolygon.geojson`](https://raw.githubusercontent.com/opengeoshub/vopendata/main/shape/multipolygon.geojson) — **12 features**, **13 polygon parts**. **A5 resolution** is shown on every frame.

## Install necessary packages

In [ ]:
%pip install vgrid geopandas matplotlib imageio pillow pya5
# optional for MP4:
%pip install imageio-ffmpeg

In [ ]:
"""Step-by-step polygon2a5_new animation (multipolygon.geojson)."""
from pathlib import Path

import a5
import geopandas as gpd
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon
from shapely.geometry import MultiPolygon

from vgrid.conversion.dggs2geo.a52geo import a52geo_u64
from vgrid.utils.geometry import check_predicate

try:
    from vgrid.conversion.vector2dggs.vector2a5 import polygon2a5_new
except ImportError:
    polygon2a5_new = None

URL = "https://raw.githubusercontent.com/opengeoshub/vopendata/main/shape/multipolygon.geojson"
RESOLUTION = 12
PREDICATE = "intersects"
COMPACT = False
OUT_GIF = "polygon2a5_new.gif"
OUT_MP4 = "polygon2a5_new.mp4"
FRAME_EVERY_N = 8
DPI = 120
A5_OPTIONS = None
SPLIT_ANTIMERIDIAN = False
PART_COLORS = ["#1f4e79", "#c55a11", "#2e7d32", "#b71c1c", "#6a1b9a", "#4e342e", "#00695c", "#5d4037"]


def cell_patches(cell_polys, facecolor, edgecolor, alpha=0.55, lw=0.4):
    patches = []
    for poly in cell_polys:
        if poly is None or poly.is_empty:
            continue
        patches.append(MplPolygon(list(poly.exterior.coords), closed=True))
    return PatchCollection(
        patches, facecolor=facecolor, edgecolor=edgecolor, alpha=alpha, linewidths=lw
    )


def polygons_from_feature(feature):
    if feature.geom_type == "Polygon":
        return [feature]
    if feature.geom_type == "MultiPolygon":
        return list(feature.geoms)
    return []


def polygons_from_gdf(gdf):
    parts = []
    for geom in gdf.geometry:
        if geom is None or geom.is_empty:
            continue
        parts.extend(polygons_from_feature(geom))
    return parts


def feature_from_gdf(gdf):
    parts = polygons_from_gdf(gdf)
    if not parts:
        raise ValueError("No polygon geometries found in input GeoJSON")
    if len(parts) == 1:
        return parts[0]
    return MultiPolygon(parts)


def part_color(part_index):
    return PART_COLORS[part_index % len(PART_COLORS)]


def render_frame(
    parts,
    active_cells,
    title,
    path,
    resolution,
    current_poly=None,
    ring=None,
    active_part=None,
):
    fig, ax = plt.subplots(figsize=(8, 8))
    union = MultiPolygon(parts) if len(parts) > 1 else parts[0]
    minx, miny, maxx, maxy = union.bounds
    pad = max(maxx - minx, maxy - miny) * 0.08 or 0.01
    ax.set_xlim(minx - pad, maxx + pad)
    ax.set_ylim(miny - pad, maxy + pad)

    for i, poly in enumerate(parts):
        color = part_color(i)
        lw = 3.0 if active_part == i else 1.8
        alpha = 1.0 if active_part is None or active_part == i else 0.45
        gpd.GeoSeries([poly]).plot(
            ax=ax, facecolor="none", edgecolor=color, lw=lw, alpha=alpha
        )
    if ring:
        lons, lats = zip(*ring)
        ax.scatter(lons, lats, c="#d62728", s=35, zorder=5)

    visited = list(active_cells) if active_cells else []
    if current_poly is not None:
        visited = [
            p
            for p in visited
            if p is not current_poly and not p.equals(current_poly)
        ]
    if visited:
        ax.add_collection(cell_patches(visited, "#2ca02c", "#1a5f1a", alpha=0.45))
    if current_poly is not None:
        ax.add_collection(
            cell_patches([current_poly], "#ffcc00", "#cc8800", alpha=0.9, lw=2.5)
        )
    ax.plot([], [], color="#ffcc00", lw=4, label="current cell")
    ax.plot([], [], color="#2ca02c", lw=4, label="cells so far")
    ax.legend(loc="upper right", fontsize=8)
    ax.text(
        0.02,
        0.98,
        f"A5 resolution: {resolution}",
        transform=ax.transAxes,
        fontsize=9,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.9),
        zorder=6,
    )
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.25)
    fig.subplots_adjust(left=0.08, right=0.92, top=0.92, bottom=0.08)
    fig.savefig(path, dpi=DPI, facecolor="white")
    plt.close(fig)


def cell_polygon(cell_id, options=None, split_antimeridian=False):
    return a52geo_u64(cell_id, options=options, split_antimeridian=split_antimeridian)


def polygon2a5_new_with_frames(
    parts,
    feature,
    resolution,
    predicate,
    frame_dir,
    compact=False,
    options=None,
    split_antimeridian=False,
):
    frame_dir.mkdir(parents=True, exist_ok=True)
    frames = []
    idx = 0
    merged_ids = []
    merged_polys = []
    seen_u64 = set()
    accumulated = []

    def snap(title, active=None, current=None, ring=None, active_part=None):
        nonlocal idx
        p = frame_dir / f"frame_{idx:04d}.png"
        render_frame(
            parts,
            active if active is not None else accumulated,
            title,
            p,
            resolution,
            current_poly=current,
            ring=ring,
            active_part=active_part,
        )
        frames.append(p)
        idx += 1

    n_parts = len(parts)
    snap(
        f"1. Input res {resolution} ({n_parts} polygon part{'s' if n_parts != 1 else ''})"
    )
    snap("2. All parts (distinct colors)")

    for part_i, polygon in enumerate(parts, start=1):
        part_idx = part_i - 1
        ring = [(lon, lat) for lon, lat in polygon.exterior.coords[:-1]]
        if len(ring) < 3:
            continue

        snap(
            f"3. Part {part_i}/{n_parts}: exterior ring ({len(ring)} vertices)",
            ring=ring,
            active_part=part_idx,
        )

        try:
            compacted_cells = a5.polygon_to_cells(ring, resolution)
        except Exception:
            compacted_cells = []

        if not compacted_cells:
            snap(
                f"4. Part {part_i}: polygon_to_cells returned no cells",
                ring=ring,
                active_part=part_idx,
            )
            continue

        discovery_polys = []
        for step, cell_id in enumerate(compacted_cells, start=1):
            cell_poly = cell_polygon(cell_id, options, split_antimeridian)
            if cell_poly is None or cell_poly.is_empty:
                continue
            discovery_polys.append(cell_poly)
            if step % FRAME_EVERY_N == 0 or step == len(compacted_cells):
                snap(
                    f"4. Part {part_i} polygon_to_cells {step}/{len(compacted_cells)}: {a5.u64_to_hex(cell_id)}",
                    active=discovery_polys,
                    current=cell_poly,
                    ring=ring,
                    active_part=part_idx,
                )

        snap(
            f"4b. Part {part_i} polygon_to_cells complete ({len(compacted_cells)} cells)",
            active=discovery_polys,
            ring=ring,
            active_part=part_idx,
        )

        try:
            candidate_cells = a5.uncompact(compacted_cells, resolution)
        except Exception:
            candidate_cells = list(compacted_cells)

        snap(
            f"5. Part {part_i}: uncompact → {len(candidate_cells)} candidates at level {resolution}",
            active=[],
            ring=ring,
            active_part=part_idx,
        )

        candidate_polys = []
        for step, cell_id in enumerate(candidate_cells, start=1):
            cell_poly = cell_polygon(cell_id, options, split_antimeridian)
            if cell_poly is None or cell_poly.is_empty:
                continue
            candidate_polys.append(cell_poly)
            if step % FRAME_EVERY_N == 0 or step == len(candidate_cells):
                snap(
                    f"5b. Part {part_i} candidate {step}/{len(candidate_cells)}: {a5.u64_to_hex(cell_id)}",
                    active=candidate_polys,
                    current=cell_poly,
                    active_part=part_idx,
                )

        part_final = []
        for cell_id in candidate_cells:
            cell_poly = cell_polygon(cell_id, options, split_antimeridian)
            if cell_poly is None or cell_poly.is_empty:
                continue
            if check_predicate(cell_poly, polygon, predicate):
                part_final.append(cell_id)
                if cell_id not in seen_u64:
                    seen_u64.add(cell_id)
                    merged_ids.append(cell_id)
                    merged_polys.append(cell_poly)
                    accumulated.append(cell_poly)

        snap(
            f"6. Part {part_i} after predicate '{predicate}' ({len(part_final)} cells)",
            active=[
                cell_polygon(cid, options, split_antimeridian) for cid in part_final
            ],
            active_part=part_idx,
        )

    final_ids = merged_ids
    final_polys = merged_polys
    if compact and merged_ids:
        try:
            final_ids = a5.compact(merged_ids)
        except Exception:
            final_ids = merged_ids
        final_polys = []
        for cell_id in final_ids:
            cell_poly = cell_polygon(cell_id, options, split_antimeridian)
            if cell_poly is not None and not cell_poly.is_empty:
                final_polys.append(cell_poly)
        snap(
            f"7. After a5.compact ({len(final_polys)} cells)",
            active=final_polys,
        )
    else:
        snap(
            f"7. Merged result ({len(final_ids)} cells across {n_parts} parts)",
            active=final_polys,
        )

    our_hex = [a5.u64_to_hex(c) for c in final_ids]
    if polygon2a5_new is not None:
        from_poly = []
        for part in parts:
            rows = polygon2a5_new(
                part,
                resolution,
                predicate=predicate,
                compact=compact,
                options=options,
                split_antimeridian=split_antimeridian,
            )
            from_poly.extend(row["a5"] for row in rows)
        if set(from_poly) != set(our_hex):
            print(
                "Warning: cell set differs from polygon2a5_new:",
                len(our_hex),
                "vs",
                len(set(from_poly)),
            )
    else:
        print("Note: polygon2a5_new not in installed vgrid; skip set verification")

    return frames, final_ids


def main():
    gdf = gpd.read_file(URL)
    parts = polygons_from_gdf(gdf)
    feature = feature_from_gdf(gdf)
    print(f"Loaded {len(gdf)} feature(s), {len(parts)} polygon part(s)")

    frame_dir = Path("_polygon2a5_new_frames")
    frames, cell_ids = polygon2a5_new_with_frames(
        parts,
        feature,
        RESOLUTION,
        PREDICATE,
        frame_dir,
        compact=COMPACT,
        options=A5_OPTIONS,
        split_antimeridian=SPLIT_ANTIMERIDIAN,
    )
    imageio.mimsave(OUT_GIF, [imageio.imread(f) for f in frames], duration=0.9)
    print(f"Wrote {OUT_GIF} ({len(frames)} frames, {len(cell_ids)} final cells)")
    try:
        writer = imageio.get_writer(OUT_MP4, fps=1.2)
        for f in frames:
            writer.append_data(imageio.imread(f))
        writer.close()
        print(f"Wrote {OUT_MP4}")
    except Exception as e:
        print(f"MP4 skipped ({e}). GIF is enough.")


if __name__ == "__main__":
    main()
